# 🦀 Day 2: Feature 2 — Advanced Data Structures (Lists, Hashes)
---

Today we extend RustKV with List and Hash value types.

- **List operations**: LPUSH, RPUSH, LPOP, RPOP, LLEN, LRANGE — O(1) push/pop with `VecDeque`
- **Hash operations**: HSET, HGET, HDEL, HGETALL, HKEYS
- **Type checking**: Error when wrong command used on wrong value type
- **Exercise**: Add LINDEX, HLEN, or HEXISTS

## Extending Value types

Use `VecDeque` for list operations — O(1) push/pop at both ends.

In [ ]:
use std::collections::{HashMap, VecDeque};

#[derive(Debug, Clone, PartialEq)]
pub enum Value {
    String(String),
    Integer(i64),
    List(VecDeque<String>),
    Hash(HashMap<String, String>),
}

// List commands
impl Value {
    pub fn lpush(&mut self, val: String) -> Result<usize, &'static str> {
        match self {
            Value::List(list) => {
                list.push_front(val);
                Ok(list.len())
            }
            _ => Err("WRONGTYPE Operation against a key holding the wrong kind of value"),
        }
    }
    pub fn rpush(&mut self, val: String) -> Result<usize, &'static str> {
        match self {
            Value::List(list) => {
                list.push_back(val);
                Ok(list.len())
            }
            _ => Err("WRONGTYPE Operation against a key holding the wrong kind of value"),
        }
    }
    pub fn lpop(&mut self) -> Result<Option<String>, &'static str> {
        match self {
            Value::List(list) => Ok(list.pop_front()),
            _ => Err("WRONGTYPE Operation against a key holding the wrong kind of value"),
        }
    }
    pub fn rpop(&mut self) -> Result<Option<String>, &'static str> {
        match self {
            Value::List(list) => Ok(list.pop_back()),
            _ => Err("WRONGTYPE Operation against a key holding the wrong kind of value"),
        }
    }
    pub fn llen(&self) -> Result<usize, &'static str> {
        match self {
            Value::List(list) => Ok(list.len()),
            _ => Err("WRONGTYPE Operation against a key holding the wrong kind of value"),
        }
    }
    pub fn lrange(&self, start: i64, stop: i64) -> Result<Vec<String>, &'static str> {
        match self {
            Value::List(list) => {
                let len = list.len() as i64;
                let start = if start < 0 { len + start } else { start };
                let stop = if stop < 0 { len + stop } else { stop };
                let start = start.max(0) as usize;
                let stop = (stop.min(len - 1) + 1).max(0) as usize;
                Ok(list.iter().skip(start).take(stop.saturating_sub(start)).cloned().collect())
            }
            _ => Err("WRONGTYPE Operation against a key holding the wrong kind of value"),
        }
    }
}

let mut list = Value::List(VecDeque::new());
list.rpush("a".to_string()).unwrap();
list.rpush("b".to_string()).unwrap();
list.lpush("z".to_string()).unwrap();
println!("LRANGE 0 -1: {:?}", list.lrange(0, -1));

In [ ]:
// Hash operations

impl Value {
    pub fn hset(&mut self, field: String, value: String) -> Result<i64, &'static str> {
        match self {
            Value::Hash(h) => {
                let is_new = !h.contains_key(&field);
                h.insert(field, value);
                Ok(if is_new { 1 } else { 0 })
            }
            _ => Err("WRONGTYPE Operation against a key holding the wrong kind of value"),
        }
    }
    pub fn hget(&self, field: &str) -> Result<Option<String>, &'static str> {
        match self {
            Value::Hash(h) => Ok(h.get(field).cloned()),
            _ => Err("WRONGTYPE Operation against a key holding the wrong kind of value"),
        }
    }
    pub fn hdel(&mut self, field: &str) -> Result<i64, &'static str> {
        match self {
            Value::Hash(h) => Ok(if h.remove(field).is_some() { 1 } else { 0 }),
            _ => Err("WRONGTYPE Operation against a key holding the wrong kind of value"),
        }
    }
    pub fn hgetall(&self) -> Result<Vec<(String, String)>, &'static str> {
        match self {
            Value::Hash(h) => Ok(h.iter().map(|(k, v)| (k.clone(), v.clone())).collect()),
            _ => Err("WRONGTYPE Operation against a key holding the wrong kind of value"),
        }
    }
    pub fn hkeys(&self) -> Result<Vec<String>, &'static str> {
        match self {
            Value::Hash(h) => Ok(h.keys().cloned().collect()),
            _ => Err("WRONGTYPE Operation against a key holding the wrong kind of value"),
        }
    }
}

let mut h = Value::Hash(HashMap::new());
h.hset("name".to_string(), "RustKV".to_string()).unwrap();
h.hset("version".to_string(), "0.1".to_string()).unwrap();
println!("HGET name: {:?}", h.hget("name"));
println!("HKEYS: {:?}", h.hkeys());

In [ ]:
// Type checking — wrong command on wrong type returns error

let mut string_val = Value::String("hello".to_string());
let mut list_val = Value::List(VecDeque::from(["a".to_string()]));

// LPUSH on String -> WRONGTYPE
let err = string_val.lpush("x".to_string());
assert!(err.is_err());

// HSET on List -> WRONGTYPE
let err = list_val.hset("f".to_string(), "v".to_string());
assert!(err.is_err());

println!("Type checking works: wrong commands return WRONGTYPE error.");

## 📝 Exercise: Add LINDEX, HLEN, or HEXISTS

- **LINDEX key index**: Return element at index (0-based, -1 = last)
- **HLEN key**: Return number of fields in hash
- **HEXISTS key field**: Return 1 if field exists, 0 otherwise

In [ ]:
// YOUR CODE HERE
// Add LINDEX, HLEN, and/or HEXISTS to the Value impl

// LINDEX: list.get(index).cloned() — handle negative indices like LRANGE
// HLEN: h.len() as i64
// HEXISTS: h.contains_key(field) -> 1 or 0

println!("Implement LINDEX, HLEN, and HEXISTS.");

## 🎯 Key Takeaways

1. Use `VecDeque` for list operations — O(1) push/pop at both ends
2. List commands: LPUSH, RPUSH, LPOP, RPOP, LLEN, LRANGE — handle negative indices
3. Hash commands: HSET, HGET, HDEL, HGETALL, HKEYS — use `HashMap<String, String>`
4. Type checking: return `WRONGTYPE` error when command doesn't match value type
5. Create List/Hash with `Value::List(VecDeque::new())` or `Value::Hash(HashMap::new())` on first write

---
**Tomorrow:** Feature 3 — Publish/Subscribe messaging